<a href="https://colab.research.google.com/github/marcouras/AI-engineering-fundamentals/blob/main/lezione4/Lezione4_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# 🤖 AI Engineering Fundamentals
## Lezione 4 — RAG: Conoscenza Personalizzata

**ITS Novitas 4.0 — Sviluppatore Intelligenza Artificiale**  
Docente: Marco Uras | 📅 Giovedì 28/05/2026

---

### 🎯 Obiettivi
- ✅ Capire la pipeline RAG completa
- ✅ Indicizzare un PDF con ChromaDB
- ✅ Implementare la ricerca semantica
- ✅ Integrare RAG nel chatbot esistente

In [ ]:
# Setup
!pip install anthropic chromadb pypdf sentence-transformers -q
from google.colab import userdata
import anthropic, os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

def chiedi_claude(domanda, system=None, max_tokens=800):
    params = {"model":"claude-haiku-4-5-20251001","max_tokens":max_tokens,
              "messages":[{"role":"user","content":domanda}]}
    if system: params["system"] = system
    return client.messages.create(**params).content[0].text

print("✅ Setup completato!")

✅ Setup completato!


---
## 1. Crea un documento di test

Per l'esercizio creiamo un documento di testo su WiData. In un progetto reale useresti un PDF vero.

In [ ]:
# Creiamo un documento di testo su WiData
documento_widata = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica e qualità dell'aria (CO2, PM2.5).
Classificazione IP67: impermeabile e resistente alla polvere. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n. Dimensioni: 85x45x30mm. Peso: 120g.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare (opzionale). Temperatura operativa: -40°C a +70°C.
Certificazioni: CE, IP65. Installazione: palo, tetto o rack.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook quando i valori superano soglie configurabili.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Machine learning per previsione anomalie e manutenzione predittiva.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptime garantito nei piani Pro ed Enterprise.

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Per informazioni commerciali: sales@widata.cloud.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
"""

# Salva su file
with open("manuale_widata.txt", "w", encoding="utf-8") as f:
    f.write(documento_widata)

print(f"✅ Documento creato: {len(documento_widata)} caratteri")

✅ Documento creato: 1846 caratteri


---
## 2. Chunking e Indicizzazione

In [ ]:
def chunka_testo(testo, chunk_size=400, overlap=50):
    """Divide il testo in chunk con overlap."""
    chunks = []
    start = 0
    while start < len(testo):
        end = start + chunk_size
        chunk = testo[start:end]
        if chunk.strip():  # ignora chunk vuoti
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunka_testo(documento_widata)
print(f"📊 Numero di chunk: {len(chunks)}")
print()
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} char) ---")
    print(chunk[:100]+"...")
    print()

📊 Numero di chunk: 6

--- Chunk 1 (400 char) ---

WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è proge...

--- Chunk 2 (400 char) ---
. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n...

--- Chunk 3 (400 char) ---
km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiFi o 4G ...

--- Chunk 4 (400 char) ---
iData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-tim...

--- Chunk 5 (400 char) ---
va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiest...

--- Chunk 6 (96 char) ---
: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
...



In [ ]:
import chromadb

# Crea il client ChromaDB in memoria
chroma_client = chromadb.Client()

# Crea o recupera la collection
collection = chroma_client.get_or_create_collection(
    name="widata_docs",
    metadata={"hnsw:space": "cosine"}  # usa similarità coseno
)

# Indicizza i chunk
collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"✅ Indicizzati {collection.count()} chunk in ChromaDB")
print("💡 ChromaDB ha calcolato automaticamente gli embedding per ogni chunk!")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 98.2MiB/s]


✅ Indicizzati 6 chunk in ChromaDB
💡 ChromaDB ha calcolato automaticamente gli embedding per ogni chunk!


---
## 3. Ricerca Semantica

In [ ]:
def cerca(domanda, n_risultati=3):
    """Cerca i chunk più rilevanti per la domanda."""
    risultati = collection.query(
        query_texts=[domanda],
        n_results=n_risultati
    )
    return risultati["documents"][0]

# Test ricerca semantica
domande_test = [
    "Quali sensori supportate per ambienti esterni?",
    "Come posso integrare i dati con il mio sistema ERP?",
    "Qual è il costo del piano professionale?",
]

for domanda in domande_test:
    print(f"\n❓ {domanda}")
    chunks_trovati = cerca(domanda, n_risultati=2)
    for i, chunk in enumerate(chunks_trovati):
        print(f"  📄 Chunk {i+1}: {chunk[:120]}...")


❓ Quali sensori supportate per ambienti esterni?
  📄 Chunk 1: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptim...
  📄 Chunk 2: iData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino...

❓ Come posso integrare i dati con il mio sistema ERP?
  📄 Chunk 1: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptim...
  📄 Chunk 2: iData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino...

❓ Qual è il costo del piano professionale?
  📄 Chunk 1: : Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
...
  📄 Chunk 2: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptim...


---
## 4. RAG Completo — Domanda + Contesto + Risposta

In [ ]:
SYSTEM_WIDATA = """
Sei l'assistente virtuale di WiData Srl, azienda IoT e smart cities di Sassari.
Rispondi SOLO basandoti sui documenti forniti nel contesto.
Se la risposta non è nei documenti, dì chiaramente: 'Non ho questa informazione nei miei documenti.'
Non inventare mai informazioni. Sii conciso e preciso.
"""

def chat_rag(domanda, n_chunks=3):
    """Chatbot con RAG: recupera contesto e genera risposta."""
    # 1. Recupera i chunk rilevanti
    chunks_rilevanti = cerca(domanda, n_risultati=n_chunks)
    contesto = "\n\n---\n\n".join(chunks_rilevanti)

    # 2. Costruisci il prompt aumentato
    prompt = f"""Documenti di riferimento:

{contesto}

---

Domanda dell'utente: {domanda}"""

    # 3. Genera la risposta
    risposta = chiedi_claude(prompt, system=SYSTEM_WIDATA)
    return risposta, chunks_rilevanti

# Test completo
domanda = "Il sensore XS200 funziona in ambienti molto freddi?"
risposta, chunks = chat_rag(domanda)

print(f"❓ {domanda}")
print(f"\n🤖 {risposta}")
print(f"\n📄 Basato su {len(chunks)} chunk")

❓ Il sensore XS200 funziona in ambienti molto freddi?

🤖 Sì, il sensore XS200 è in grado di funzionare in ambienti freddi. 

Secondo le specifiche tecniche, il sensore misura temperature da **-20°C a +60°C**, quindi può operare correttamente anche in condizioni di freddo intenso fino a -20°C.

Inoltre, grazie alla classificazione **IP67**, il sensore è impermeabile e resistente alla polvere, il che lo rende adatto anche ad ambienti difficili dal punto di vista climatico.

📄 Basato su 3 chunk


In [ ]:
# Test con domanda fuori dai documenti
domanda_off = "Quali sono i migliori smartphone del 2025?"
risposta_off, _ = chat_rag(domanda_off)
print(f"❓ {domanda_off}")
print(f"\n🤖 {risposta_off}")
print("\n💡 Il sistema dovrebbe rifiutarsi di rispondere!")

❓ Quali sono i migliori smartphone del 2025?

🤖 Non ho questa informazione nei miei documenti.

I documenti che ho a disposizione riguardano esclusivamente i servizi e prodotti di WiData Srl, come piani di abbonamento, sensori IoT, gateway e piattaforme di analisi dati. Non contengono informazioni su smartphone.

Posso aiutarti con domande relative ai nostri prodotti e servizi IoT. Desideri saperne di più?

💡 Il sistema dovrebbe rifiutarsi di rispondere!


---
## ⭐ Esercizi

In [ ]:
NOME_STUDENTE = ""  # ← SCRIVI IL TUO NOME
if NOME_STUDENTE:
    print(f"✅ Notebook di: {NOME_STUDENTE}")
else:
    print("⚠️ Scrivi il tuo nome!")

### Esercizio 1 — Indicizza un documento tuo ★☆☆
Crea un documento di testo su un argomento a tua scelta (può essere anche una dispensa del corso, una ricetta, un regolamento). Indicizzalo in ChromaDB e fai 3 domande. I chunk recuperati sono rilevanti?

In [ ]:
# ESERCIZIO 1
mio_documento = """
# Scrivi qui il tuo documento (almeno 300 caratteri)
"""

# Crea una nuova collection
# mia_collection = chroma_client.get_or_create_collection(name="mio_doc")

# Chunka e indicizza
# chunks = chunka_testo(mio_documento)
# mia_collection.add(...)

# Fai 3 domande e stampa i chunk recuperati
# ...

### Esercizio 2 — Sperimenta con il chunking ★★☆
Prova a indicizzare lo stesso documento con chunk_size=200, 400 e 800. Per la stessa domanda, i chunk recuperati sono diversi? Quale dimensione dà risultati migliori?

In [ ]:
# ESERCIZIO 2
domanda_test = "Inserisci qui una domanda sul tuo documento"  # ← modifica

for chunk_size in [200, 400, 800]:
    print(f"\n{'='*50}")
    print(f"chunk_size = {chunk_size}")
    print('='*50)

    # TODO: crea collection, chunka con dimensione diversa, indicizza, cerca
    # chunks = chunka_testo(mio_documento, chunk_size=chunk_size)
    # ...
    # risultati = cerca(domanda_test)
    # stampa i risultati
    pass

# Commento: quale chunk_size ha dato i risultati migliori?
# Risposta: ...

### Esercizio 3 — RAG + storia conversazione ★★☆
Integra RAG nella funzione `chat()` della Lezione 3 (quella con la history). Ogni risposta deve usare sia il contesto RAG che la storia della conversazione.

In [ ]:
# ESERCIZIO 3
history = []

def chat_rag_con_storia(domanda):
    """Chatbot con RAG + conversazione multi-turno."""
    # TODO:
    # 1. Recupera chunk rilevanti con cerca()
    # 2. Costruisci il messaggio con contesto + domanda
    # 3. Aggiungi alla history
    # 4. Chiama l'API con tutta la history
    # 5. Aggiungi la risposta alla history
    # 6. Restituisci la risposta
    pass

# Test: domande collegate che richiedono sia RAG che memoria
# chat_rag_con_storia("Parlami del sensore XS200")
# chat_rag_con_storia("Qual è la sua autonomia?")
# chat_rag_con_storia("Costa molto?")

### Esercizio 4 — Chatbot RAG WiData completo ★★★ (Deliverable!)

Costruisci il chatbot completo con:
- RAG sul documento WiData
- Conversation history (sliding window)
- Streaming
- System prompt WiData con istruzione anti-hallucination
- Loop interattivo con `input()`
- Stampa i chunk usati per ogni risposta (per debug)

In [ ]:
# ESERCIZIO 4 — Chatbot RAG completo (DELIVERABLE)
import chromadb, json, os
from google.colab import userdata
import anthropic

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

SYSTEM = """
# ← Completa il system prompt WiData con istruzione anti-hallucination
"""

MAX_MESSAGGI = 10

def setup_rag(testo):
    """Indicizza il documento e restituisce la collection."""
    pass  # ← implementa

def cerca(domanda, collection, n=3):
    """Ricerca semantica nella collection."""
    pass  # ← implementa

def chat_completo(domanda, history, collection):
    """Chatbot con RAG + storia + streaming."""
    pass  # ← implementa

def main():
    collection = setup_rag(documento_widata)
    history = []
    print("🤖 Chatbot WiData RAG avviato. Digita 'esci' per uscire.\n")

    while True:
        utente = input("Tu: ")
        if utente.lower() == "esci":
            print("👋 Arrivederci!")
            break
        chat_completo(utente, history, collection)

# main()  # Decommentare per eseguire

---
## 📤 Consegna

1. Completa tutti gli esercizi
2. Scarica: `File → Scarica → .ipynb`
3. Rinomina: `Lezione4_TUONOME.ipynb`
4. Carica su GitHub in `lezione4/`

```bash
git add lezione4/
git commit -m "Lezione 4 completata"
git push
```

---
### 📖 Per la prossima lezione (Giovedì 04/06)
Leggi **Huyen Cap. 6 — sezione Agents**

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*